In [1]:
!pip install langchain_community langchain_openai faiss-cpu pypdf
!pip install python-dotenv
import os 
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# 환경 변수 가져오기
API_KEY = os.getenv("API_KEY")

In [2]:
import os
import openai
from langchain.chains import AnalyzeDocumentChain
from langchain.chains.question_answering import load_qa_chain
from langchain.chat_models import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQAWithSourcesChain, LLMChain, StuffDocumentsChain
# from langchain.retrievers import EmbeddingRetriever
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.docstore.document import Document
import numpy as np
####
from dotenv import load_dotenv
import json
import faiss
from openai import OpenAI


#### api 키 설정
api_key = API_KEY
os.environ['OPENAI_API_KEY'] = api_key
####

client = OpenAI(
    api_key=api_key,
)

def generate_text(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model="o1",
        messages=[
            {"role":"system", "content":system_prompt},
            {"role": "user", "content": user_prompt}
            ]
    )
    return response.choices[0].message.content.strip()

## 논점 프롬프트 및 생성

In [3]:
k_system_text = '''
당신은 대한민국 수학능력시험 국어영역 독서 과목의 지문을 출제하는 한국교육과정평가원 출제위원이다. 
고등학교 3학년 수준의 수험생을 평가할 수 있는 지문을 아래의 핵심 논점 및 난이도 요구사항, 금지사항을 반영하여 작성하십시오.

**핵심 논점 및 난이도 요구사항**
- 주요 개념 간의 관계를 논리적으로 설명하고, 지문에 나타난 논지의 타당성을 검토할 것
- 동일한 화제에 대한 상반되거나 다양한 관점을 비교·분석하며, 각 관점의 타당성을 비판적으로 평가할 수 있도록 구성할 것
- 각 분야의 학문적 배경을 반영하되, 개념적 깊이를 확보하고, 전문 용어는 문맥 속에서 명확히 설명할 것
- 단순한 정보 전달이 아니라 수험생이 논리적 추론을 수행할 수 있도록 유도할 것

**금지 사항** 
- 모호하거나 중의적인 표현 사용 금지
- 문학 작품 생성 금지
- 허구적 사건이나 인물명 사용 금지
- 비문 생성 금지 - (가), (나), (다) 등의 기호 사용 금지
- 결론에서 전체 내용을 요약하거나 교훈 제시 금지
- 자극적이거나 선정적인 문체 사용 금지
- 특정 집단을 비하하거나 옹호하는 내용 또는 잘못된 고정관념을 유발하는 내용 작성 금지

위 조건을 철저히 준수하여 수학능력시험 국어영역 독서 과목 지문을 출제하십시오.'''

In [4]:
passage_result = "9세기 초 프랑스의 미술계에서는 신고전주의가 유행하였다. 고대 그리스나 로마 시절의 모티프를 사용하고 고고학적 발견에 따른 정확한 묘사를 추구했던 신고전주의는 엄격하고 균형적인 구도, 명확한 윤곽 등의 형식적 특성을 추구하였다. 하지만 곧 새로운 양식이 나타났는데, 풍부한 색채, 격정적인 표현 등을 중시하는 낭만주의였다. 이후 프랑스에서는 신고전주의와 낭만주의가 대립하면서도 서로 영향을 주고받는 상황이 이어졌다. 당시 프랑스 미술의 중심은 정부에서 주관하는 ‘살롱전’이었는데, 신고전주의와 낭만주의의 보수적인 화풍에 불만을 가지고 새로운 형태의 그림을 그린 프랑스의 젊은 화가들은 여기에 초대될 수 없었다. 그래서 그들은 자신들의 전시회를 따로 열었다. 전시회에는 모네의 <인상: 해돋이>라는 그림이 걸려 있었는데, 그 그림을 본 비평가들은 혹평을 하며 그림의 제목을 따서 전시회에 참여한 화가들을 ‘인상주의자’라고 조롱하였고, 여기에서 탄생한 용어가 인상주의이다. 인상주의는 19세기 후반을 대표하는 예술 사조로, 기존의 미술이 추구했던 역사, 종교 등의 소재를 벗어나서 당시 사람들의 일상적인 모습이나 자연의 풍경을 소재로 삼았다. 특히 빛의 효과에 주목하였는데, 인상주의 화가들은 빛에 따라 대상이 시시각각 변화하는 순간을 포착하여 대상을 눈에 보이는 그대로 표현하였다. 인상주의 화가들은 빛을 통해 사람의 눈에 들어온 대상의 모습이 상황이나 조건에 따라 제각기 달라질 수 있기 때문에, 대상을 형식적으로 고정하여 표현하면 실제 대상의 모습과 달라질 수 있다고 생각하였다. 그들은 색채 또한 대상의 고유색보다는 빛에 의해 변하는 색을 추구하였다. 예를 들어, 태양을 그리더라도 붉은색이라는 관념 속의 색으로만 표현하는 것이 아니라, 관찰 순간 눈에 보이는 색을 그대로 표현하는 것이다. 또한 짧은 붓질이나 두드러지지 않는 명암을 사용해서 색채를 구성하여 대상의 윤곽을 뚜렷하지 않게 표현하였다. 이는 인상주의 화가들이 대상을 일반적인 관념 속에서 정밀하고 분명하게 묘사하기보다는 순간적으로 보이는 모습을 중시했음을 보여 준다. 이러한 인상주의의 특징은 대상뿐만 아니라 배경과 순간적인 느낌을 한 번에 포착하여 이를 화폭에 담기 위한 노력으로 볼 수 있다. 따라서 그림에서 대상은 모호해질 수밖에 없었고, 중요한 것은 화가가 그 순간 눈을 통해 보고 있는 감각적인 느낌이 어떻게 감상자에게 잘 전달되느냐 하는 것이었다. 인상주의는 감성적인 표현을 중시하면서 세계를 더욱 정확하게 기록하려고 하였고, 이는 음악이나 문학 등의 다른 예술 분야에까지 영향을 미쳤다. 19세기 유럽을 지배했던 음악 사조는 독일에서 발생한 낭만주의였다. 낭만주의는 청중을 압도하는 과장된 표현, 구체적인 대상이나 사상을 나타내는 표제의 사용, 소나타 형식 등의 특징이 있었으며, 교향곡*의 양식을 완성하고 있었다. 하지만 프랑스 음악가들은 이러한 낭만주의의 구조적 특성이 프랑스와는 맞지 않는다고 생각하였고, 진정한 프랑스 음악을 찾기 위해 시도하였다. 프랑스의 음악가인 드뷔시는 음악은 규칙보다는 즐거움을 따라야 한다고 말하며, 이를 위해 감각적인 환상이 필요하다고 하였는데, 이는 당시 프랑스에서 유행하던 미술 사조인 인상주의와 일맥상통하는 면이 있었다. 드뷔시나 라벨 등의 음악가들은 음악을 통해 순간의 인상을 표현하고, 지성보다는 감성으로 청중을 매료시키고자 하였는데, 이러한 프랑스 음악의 경향을 인상주의 음악이라고 부른다."

In [16]:
k_user_prompt = f"""
다음은 한국교육과정평가원 스타일로 생성된 수능 독서 지문입니다.

[생성된 지문]
{passage_result}

이 지문에서 학생이 반드시 이해해야 할 핵심 논점 2~3개를 요약하세요.
각 논점은 1~2문장으로 정리하고, 출제 의도를 반영해야 합니다.
또한, 글의 핵심 내용을 15자 이내로 요약한 키워드를 1~3개 뽑아주세요.(반드시 지문에 포함된 키워드여야 합니다.) 쉼표(,)로 구분해주세요.

출력 형식:
논점: 
1. 논점 1
2. 논점 2
키워드:  키워드1, 키워드2, 키워드3
"""

In [17]:
results = generate_text(k_system_text, k_user_prompt)

# 줄바꿈 기준으로 분리
lines = results.split("\n")

# 논점과 키워드 저장할 리스트
key_points = []
keywords = None

for line in lines:
    if line.startswith("1.") or line.startswith("2.") or line.startswith("3."):
        key_points.append(line.strip())
    elif line.startswith("키워드:"):
        keywords = line.replace("키워드:", "").strip()

# 기본값 설정 (예외 처리)
key_points = key_points if key_points else ["논점 없음"]
keywords = keywords if keywords else "키워드 없음"

# 최종 출력
print("논점:")
for point in key_points:
    print(f"{point}")

print(f"키워드: {keywords}")


논점:
- 1. 신고전주의와 낭만주의가 서로 대립하면서도 영향을 주고받는 과정에서, 기존 화풍에 부정적인 반응을 보인 젊은 화가들이 ‘인상주의’라는 새로운 양식을 탄생시켰음을 이해해야 한다.
- 2. 인상주의는 순간적으로 보이는 대상과 빛의 효과를 중시하여 고정된 관념보다는 감각적이고 가변적인 이미지를 표현하며, 이는 회화뿐만 아니라 음악 분야에도 영향을 미쳤음을 파악해야 한다.
키워드: 인상주의, 낭만주의, 드뷔시


## 문항 프롬프트 및 생성

In [9]:
q_system_text = '''
    당신은 대한민국 대학수학능력시험의 국어 영역 독서 분야 문항을 생성하는 시험출제 전문가이다. 
    문항은 지문에서 측정하고자 하는 내용을 정확히 반영하고, 핵심 내용을 간결하고 구조적이며 체계적으로 구성해야 한다. 

    선택지를 작성할 때는 문법적, 논리적으로 지문과 일치하도록 하며, 정답과 오답이 명확하게 구별되도록 해야한다.
    정답의 위치는 무작위로 배치하여 특정 패턴이 드러나지 않도록 하고, 다른 문항과 중복되지 않도록 해야 한다. 
    또한 단순히 특정 어휘를 대체하는 방식으로 오답을 구성하면 안된다. 
''' 

In [10]:
# 1. 디렉토리 내의 모든 문제 관련 .txt 파일을 불러오기
loader_question = DirectoryLoader("./RAG자료/문제 관련", glob="*.txt", loader_cls=TextLoader)
documents_question = loader_question.load()

# 3. OpenAI 임베딩 모델 초기화
embeddings_model = OpenAIEmbeddings()

# 4.1 문제 전체 문서 FAISS 인덱스 생성
texts_question = [doc.page_content for doc in documents_question]
metadata_question = [doc.metadata for doc in documents_question]
vector_db_question_all = FAISS.from_texts(texts_question, embeddings_model, metadatas=metadata_question)

# 5. FAISS 인덱스 저장 (문항 전체)
faiss_question_all_path = "./faiss_index/faiss_index_question_all"
faiss_question_required_path = "./faiss_index/faiss_index_question_required"

vector_db_question_all.save_local(faiss_question_all_path)
print(f"문제 전체 문서 FAISS 벡터 DB 저장 완료: {faiss_question_all_path}")

문제 전체 문서 FAISS 벡터 DB 저장 완료: ./faiss_index/faiss_index_question_all


In [15]:
# 전체 문서 FAISS에서 유사한 문서 3개 검색
question_guildlines = vector_db_question_all.similarity_search(subject_query, k=1)

# 검색 결과 출력
for i, res in enumerate(question_guildlines):
    print(f"\n[{i+1}] 검색된 문서:")
    print("-" * 50)
    print(res.page_content)
    print("-" * 50)
    print("📄 Metadata:", res.metadata)



[1] 검색된 문서:
--------------------------------------------------
- 기술의 핵심 원리나 방법을 <보기>의 그림으로 제시하여, 장치나 시스템의 작동 원리에 대해 추론을 할 수 있는지를 평가히는 문항이 자주 출제되고 있다.
--------------------------------------------------
📄 Metadata: {'source': 'RAG자료\\문제 관련\\26수특_기술분야의 출제경향_문항.txt'}


In [16]:
question_type =""

#부정형/정답형 처리
if "않은" in question_type or "않는" in question_type:
    narrative_style = "부정형"
else:
    narrative_style = "정답형"

In [17]:
q_user_prompt = f"""
    다음은 문항과 선지를 작성할 때 반드시 고려해야 할 기준이다. 문항과 선지를 작성하는 지침에 따라 내용을 참고하여 정확히 반영해라. 아래 지문을 바탕으로 5개의 선지로 이루어진 문항을 1개 작성해라.

    [지문]
    {passage_result}

    [핵심 논점(출제 의도)]
    {key_points}

    [문항 유형]
    ## 유형 종류
    1. '사실적 읽기'유형의 문항은 글의 내용을 정확히 파악하고 이해하는 능력, 글에 담긴 정보 간의 관계를 파악하는 능력, 글의 조직과 구조를 파악 이해할 수 있는 능력을 측정하는 문항으로 구성되어야 한다.
    2. '추론적 읽기'유형이란 글에 직접 명시되지 않은 정보를 논리적으로 추론하는 능력, 내포적(함축적, 문맥적) 의미를 추론하는 능력, 전제나 논거를 추론하는 능력, 전후 관계를 추론하는 능력, 필자의 견해,주장,의도를 추론하는 능력을 측정하는 문항으로 구성 한다.
    3. '비판적 읽기'유형의 문항은 글의 내용을 비판적으로 이해하고 그 내용의 타당성, 적절성, 가치 및 우열에 대해 평가하는 능력을 측정하는 문항으로 구성되어야 한다.
    4. '어휘 및 문법'유형은 정확하고 효과적인 어휘 사용 능력과 기본 개념의 이해 능력을 측정 요소로 한다. 어휘의 지시적,문맥적,비유적 의미를 이해하고 표현하는 능력, 문장과 문단의 의미가 분명하게 드러나도록 표현하는 능력을 평가할 수 있어야 한다.

    [유형별 예시 데이터]
    긍정형
    #### a. 사실적 읽기
    - 윗글에 대한 이해로 적절한 것은?
    - 윗글의 내용 전개 방식으로 적절한 것은?
    - 윗글을 읽고 보인 반응으로 적절한 것은?
    - 글에서 알 수 있는 'OOO'의 생각으로 적절한 것은?
    - 'OOO'에 해당하는 내용으로 적절한 것은?

    #### b. 추론적 읽기
    - 윗글을 바탕으로 'OOO'를 이해한 내용으로 적절한 것은?
    - 필자의 관점에서 'OOO'를 평가한 내용으로 적절한 것은?
    - 필자의 관점에서 'OOO'를 이해한 내용으로 적절한 것은?
    - 윗글을 읽고 추론한 내용으로 적절한 것은?

    #### c. 비판적 읽기
    - 윗글을 읽고 'OOO'에 대해 보인 반응으로 적절한 것은?
    - 'OOO'를 바탕으로 윗글에 대한 반응으로 적절한 것은?

    ### d. 어휘 및 문법
    - 문맥상 'OOO'과 바꾸어 쓰기에 적절한 것은?

    부정형
    #### a. 사실적 읽기
    - 윗글에 대한 이해로 적절하지 않은 것은?
    - 윗글의 내용 전개 방식으로 적절하지 않은 것은?
    - 윗글을 읽고 보인 반응으로 적절하지 않은 것은?
    - 글에서 알 수 있는 'OOO'의 생각으로 적절하지 않은 것은?
    - 'OOO'에 해당하는 내용으로 적절하지 않은 것은?

    #### b. 추론적 읽기
    - 윗글을 바탕으로 'OOO'를 이해한 내용으로 적절하지 않은 것은?
    - 필자의 관점에서 'OOO'를 평가한 내용으로 적절하지 않은 것은?
    - 필자의 관점에서 'OOO'를 이해한 내용으로 적절하지 않은 것은?
    - 윗글을 읽고 추론한 내용으로 적절하지 않은 것은?
    - 'OOO'의 이유로 적절하지 않은 것은?

    #### c. 비판적 읽기
    - 윗글을 읽고 'OOO'에 대해 보인 반응으로 적절하지 않은 것은?

    ### d. 어휘 및 문법
    - 문맥상 'OOO'과 바꾸어 쓰기에 적절하지 않은 것은?

    다음은 선지 구성 및 문항 작성 원칙에 대한 원칙이다.
    출제 오류가 발생하지 않도록, 선지는 모두 지문 내용이나 논리와 명백히 연결되어야 한다. 문항이 비논리적이거나 두 개 이상의 답이 나올 수 있으면 안 된다. 문항과 정답이 명확하지 않은 경우 생성을 금지한다. 지문 안에서 하나의 정보를 읽고 풀 수 있는 선지가 2개 이상으로 구성되면 안 된다. 지문에 쓰인 명사는 풀어쓰거나 비슷한 표현으로 바꾸지 않고 그대로 써야 한다. 선지의 문장은 다음 예시처럼 길이에 따라 짧은 문장에서 긴 문장 순으로 배열해야 한다.

    다음은 선지의 예시이다. 
    1. 밑줄 긋기는 독자의 기억을 돕는다. 
    2. 무분별한 밑줄 긋기는 독서 흐름을 방해할 수 있다. 
    3. 특정 정보를 강조하여 시각적 주의를 기울이도록 한다. 
    4. 너무 많은 정보를 표시하면 중요한 내용을 파악하기 어려워질 수 있다.
    5. 효과적인 밑줄 긋기 방법은 핵심 정보만 표시하고 과도한 사용을 자제하는 것이다.

    [문항 유형]
    질문 : {question_type}
    서술 방식 : {narrative_style}
    해설:(정답의 근거와 오답이 틀린 이유를 포함한 상세 해설 최소 100자 최대 200자로 작성하라)
    다음은 문항 생성 출력 형식이다. 출력 시 list 형태로 담아서 str로 출력한다.

    {
    "generated_question":"질문",
    "generated_option":"["","","","",""]",
    "generated_answer":"번호",
    "generated_description":"해설"
    }

    
    """


In [ ]:
question_data = generate_text(q_system_text,q_user_prompt)
question_data